# UrbanEye: Air Pollution AI Model Training

In this notebook, we transform the **Air Pollution Dataset** into a trained Machine Learning model (`scikit-learn` Random Forest). 
The model will predict the **Air Quality Index (AQI)** based on the measured pollutant levels. Once trained and exported, the FastAPI backend will load this model to make real-time predictions for the Interactive Dashboard.

## 1. Import Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import joblib
import os

import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")

## 2. Data Loading & Exploratory Data Analysis (EDA)

In [ ]:
# Define paths
DATA_PATH = r"H:\final year project\Air Pollution Dataset\city_day.csv"
MODEL_EXPORT_PATH = r"H:\final year project\backend\ai_models\pollution_model.pkl"

# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with {df.shape[0]} records and {df.shape[1]} columns.\n")

display(df.head())

In [ ]:
# Understand dataset statistics
display(df.describe())

In [ ]:
# Check for missing values
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap before Preprocessing')
plt.show()

## 3. Data Preprocessing & Feature Engineering

In [ ]:
# The features we will use for prediction
FEATURES = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
TARGET = 'AQI'

# Clean Data: drop rows where TARGET (AQI) or all features are entirely missing
df_clean = df.dropna(subset=[TARGET]).copy()

# Advanced Imputation: For missing feature values, we will fill with the median of that specific column
# Median is chosen over mean as it is more robust to outliers in pollution data
for feature in FEATURES:
    df_clean[feature].fillna(df_clean[feature].median(), inplace=True)

print(f"Data points after cleaning: {len(df_clean)}")

In [ ]:
# Visualize Correlation Matrix to see which pollutants affect AQI most
plt.figure(figsize=(14, 10))
correlation_matrix = df_clean[FEATURES + [TARGET]].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Pollutants vs AQI')
plt.show()

## 4. Model Training (Random Forest Regressor)

In [ ]:
# Split into training and test sets
X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} samples, testing on {len(X_test)} samples...\n")

# Train a Random Forest Regressor
# We limit max_depth and estimators to speed up training on CPU while avoiding overfitting
model = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

print("Training Complete!")

## 5. Model Evaluation Metrics and Visualizations

*Note: Because predicting AQI is a **Regression** problem (predicting a continuous number), metrics like Accuracy, Precision, and Recall are typically meant for **Classification** (predicting categories like 'Good', 'Bad').*

*To show regression performance professionally, we use **R² Score**, **MAE**, and **RMSE**. However, to provide a proxy for 'Accuracy', we will calculate a Custom Regression Accuracy metric based on a tolerance threshold.*

In [ ]:
# Predictions on test set
predictions = model.predict(X_test)

# --- 1. Standard Regression Metrics ---
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"--- Standard Metrics ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} AQI points")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} AQI points")
print(f"R-squared Score (R²): {r2:.4f} (Closer to 1.0 is better)\n")

# --- 2. Custom Classification-like Accuracy (Tolerance Based) ---
# Let's say a prediction is 'accurate' if it is within ± 20 AQI points of the real value
tolerance = 20
accurate_predictions = np.abs(predictions - y_test) <= tolerance
custom_accuracy = (accurate_predictions.sum() / len(y_test)) * 100

print(f"--- Custom Accuracy Metric ---")
print(f"Accuracy (within ± {tolerance} AQI points): {custom_accuracy:.2f}%")

In [ ]:
# Visualization 1: Actual vs Predicted AQI
plt.figure(figsize=(10, 6))
plt.scatter(y_test, predictions, alpha=0.5, color='blue', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual AQI')
plt.ylabel('Predicted AQI')
plt.title('Actual vs Predicted AQI scatter plot')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Feature Importance
feature_importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=feature_importances.index, palette='viridis')
plt.title('Feature Importances in Predicting AQI')
plt.xlabel('Relative Importance')
plt.ylabel('Pollutants')
plt.show()

## 6. Export Model for the Backend API

In [ ]:
# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(MODEL_EXPORT_PATH), exist_ok=True)

# Save the trained model to a `.pkl` file so FastAPI can load it
joblib.dump(model, MODEL_EXPORT_PATH)

print(f"✅ Model successfully exported to: {MODEL_EXPORT_PATH}")